# DEC - Saarland

In [ ]:
import os
from glob import glob

from tqdm import tqdm
import zipfile
import pandas as pd
import pyproj
from camelsp import get_metadata

from camels_de1h import get_input_path, get_nuts_id_from_provider_id, Station1h

## Extract data

In [2]:
input_path = get_input_path("DEC")

# extract zip
if not (input_path / "Q_und_W").exists():
    os.makedirs(input_path / "Q_und_W")
    with zipfile.ZipFile(input_path / "camels_de_v1.zip", "r") as zip_ref:
        zip_ref.extractall(input_path / "Q_und_W")
        print("Data extracted.")
else:
    print("Data already extracted.")

Data already extracted.


## Parse data

**Timezone: UTC+1**

Data is in hourly resolution (not 15 minutes).

In [9]:
ids_q = [path_q.name.split("-")[0] for path_q in (input_path / "Q_und_W" / "Stundenmittel_Q").glob("*.lila")]
ids_w = [path_w.name.split("-")[0] for path_w in (input_path / "Q_und_W" / "Stundenmittel_W").glob("*.lila")]

if set(ids_q) == set(ids_w):
    ids = ids_q
else:
    raise ValueError("IDs in Q and W do not match.")

Read metadata. We use the metadata file from CAMELS-DE daily (`Qry_Pegel-Stammdaten.xlsx`), as this contains all the stations provided with hourly data, the metadata file that came with the hourly data (`Pegel_SL.xlsx`) is not complete.

There were some errors in the metadata from CAMELS-DE, these errors are corrected manually in the cell below.

In [55]:
raw_meta_all_camelsde = pd.read_excel("/home/alexd/Projekte/CAMELS/Github/camelsp/input_data/Q_and_W/SL_Saarland/Qry_Pegel-Stammdaten.xlsx", decimal=",")

raw_meta_all_camelsde["MSTNR"] = raw_meta_all_camelsde["MSTNR"].astype(str)

# make e.g. 100,2km² to 100.2
raw_meta_all_camelsde["EZG_Gr"] = raw_meta_all_camelsde["EZG_Gr"].str.replace(",", ".").str.replace("km²", "").astype(float)

# make e.g. 344,48 m über NN / m ü. NN to 344.48, comma can also be a point already
raw_meta_all_camelsde["PNP"] = raw_meta_all_camelsde["PNP"].str.split(" ").str[0].str.replace(",", ".").astype(float)

# Pegel Weiskirchen (1411120) has wrong catchment area
raw_meta_all_camelsde.loc[raw_meta_all_camelsde["MSTNR"] == "1411120", "EZG_Gr"] = 4.2

# Pegel Oberleuken (1421120) has wrong Rechtswert
raw_meta_all_camelsde.loc[raw_meta_all_camelsde["MSTNR"] == "1421120", "RW"] = 2533990

# Pegel Nohfelden (1231120) has wrong Rechtswert
raw_meta_all_camelsde.loc[raw_meta_all_camelsde["MSTNR"] == "1231120", "RW"] = 2582697

# Pegel Perl (1142120) has wrong Rechtswert and Hochwert
raw_meta_all_camelsde.loc[raw_meta_all_camelsde["MSTNR"] == "1142120", "RW"] = 2526790
raw_meta_all_camelsde.loc[raw_meta_all_camelsde["MSTNR"] == "1142120", "HW"] = 5481811

for id in ids:
    if id not in raw_meta_all_camelsde["MSTNR"].values:
        print(f"ID {id} not in meta data.")
        continue

In [132]:
def parse_station_data(id: str) -> [pd.DataFrame, pd.DataFrame]:
    fname_q = [f for f in input_path.glob(f"Q_und_W/Stundenmittel_Q*/{id}-Q.lila")][0]
    fname_w = [f for f in input_path.glob(f"Q_und_W/Stundenmittel_W*/{id}-W.lila")][0]

    header_q = pd.read_csv(fname_q, sep=";", nrows=7, header=None, encoding="latin1", usecols=[0, 1])
    data_q = pd.read_csv(fname_q, sep=";", skiprows=7, header=None, encoding="latin1", na_values=[-9999], 
                        usecols=[0, 1], parse_dates=["date"], names=["date", "discharge_vol_obs"], date_format="%d.%m.%Y %H:%M")

    header_w = pd.read_csv(fname_w, sep=";", nrows=7, header=None, encoding="latin1", usecols=[0, 1])
    data_w = pd.read_csv(fname_w, sep=";", skiprows=7, header=None, encoding="latin1", na_values=[-9999],
                        usecols=[0, 1], parse_dates=["date"], names=["date", "water_level_obs"], date_format="%d.%m.%Y %H:%M")

    # for station 1013120, there are a few periods where water level has a constant value of 321, which cannot be correct as it is higher than all other values for the station and Q is NaN for these periods
    if id == "1013120":
        # replace the constant value with nan
        data_w.loc[data_w["water_level_obs"] == 321, "water_level_obs"] = pd.NA
        # also replace values==0 (happens mostly around the same time as the 321 values)
        data_w.loc[data_w["water_level_obs"] == 0, "water_level_obs"] = pd.NA

    # check Station and Stationsnummer of header_q and header_w are the same
    assert header_q.iloc[0].equals(header_w.iloc[0]) # Station
    assert header_q.iloc[1].equals(header_w.iloc[1]) # Stationsnummer

    # merge the headers
    header = header_q.T
    header.columns = header.iloc[0]
    header = header.drop(header.index[0])
    header["Datenart"] = f"{header_q.iloc[2][1]}, {header_w.iloc[2][1]}"
    header["Dimension"] = f"{header_q.iloc[3][1]}, {header_w.iloc[3][1]}"


    # merge data
    data = data_q.merge(data_w, on="date", how="outer")

    # fill potential gaps with NaN
    data = data.set_index("date").resample("1h").asfreq().reset_index()

    # convert to utc+0
    data["date"] = data.set_index("date").index.tz_localize("Etc/GMT-1").tz_convert("UTC")

    return header, data

In [88]:
id = ids[0]

fname_q = [f for f in input_path.glob(f"Q_und_W/Stundenmittel_Q*/{id}-Q.lila")][0]
fname_w = [f for f in input_path.glob(f"Q_und_W/Stundenmittel_W*/{id}-W.lila")][0]
header_q = pd.read_csv(fname_q, sep=";", nrows=7, header=None, encoding="latin1", usecols=[0, 1])
data_q = pd.read_csv(fname_q, sep=";", skiprows=7, header=None, encoding="latin1", na_values=[-9999], 
                     usecols=[0, 1], parse_dates=["date"], names=["date", "discharge_vol_obs"], date_format="%d.%m.%Y %H:%M")

header_w = pd.read_csv(fname_w, sep=";", nrows=7, header=None, encoding="latin1", usecols=[0, 1])
data_w = pd.read_csv(fname_w, sep=";", skiprows=7, header=None, encoding="latin1", na_values=[-9999],
                     usecols=[0, 1], parse_dates=["date"], names=["date", "water_level_obs"], date_format="%d.%m.%Y %H:%M")

# check header_q and header_w are the same except Datenart and Dimension
assert header_q[header_q.index.isin([0,1,4,5,6])].equals(header_w[header_w.index.isin([0,1,4,5,6])])

# merge data
data = data_q.merge(data_w, on="date", how="outer")

# convert to utc+0
data["date"] = data.set_index("date").index.tz_localize("Etc/GMT-1").tz_convert("UTC")

data

,date,discharge_vol_obs,water_level_obs
0,1973-10-31 23:00:00+00:00,0.024,6
1,1973-11-01 00:00:00+00:00,0.024,6
2,1973-11-01 01:00:00+00:00,0.024,6
3,1973-11-01 02:00:00+00:00,0.024,6
4,1973-11-01 03:00:00+00:00,0.024,6
...,...,...,...
443395,2024-05-31 18:00:00+00:00,0.107,16
443396,2024-05-31 19:00:00+00:00,0.107,16
443397,2024-05-31 20:00:00+00:00,0.107,16
443398,2024-05-31 21:00:00+00:00,0.107,16


In [133]:
for id in tqdm(ids):
    # get the gauge_id from the nuts_mapping, add it if it does not exist
    gauge_id = get_nuts_id_from_provider_id(id, "DEC", add_missing=True)

    # parse data for the station
    header, data = parse_station_data(id)

    # get the metadata of the station
    raw_meta = raw_meta_all_camelsde[raw_meta_all_camelsde["MSTNR"] == id]

    # get the metadata of the station in camelsp to see if the station was part of camelsp
    meta_camelsp = get_metadata()
    meta_camelsp = meta_camelsp[meta_camelsp["provider_id"] == id]

    if not raw_meta.empty:
        # parse the location
        easting, northing = raw_meta["RW"].values[0], raw_meta["HW"].values[0]

        # from EPSG:25832 to EPSG:4326
        transformer = pyproj.Transformer.from_crs("epsg:31466", "epsg:4326", always_xy=True)
        lon, lat = transformer.transform(easting, northing)

        # from EPSG:31466 to EPSG:3035
        transformer = pyproj.Transformer.from_crs("epsg:31466", "epsg:3035", always_xy=True)
        easting, northing = transformer.transform(easting, northing)

        # if the station is not in the camelsp metadata, use the raw metadata
        station_meta = {
            "gauge_id": gauge_id,
            "provider_id": id,
            "gauge_name": raw_meta["MSTBEM"].values[0],
            "waterbody_name": raw_meta["Gewässer"].values[0],
            "federal_state": "Saarland",
            "gauge_lon": lon,
            "gauge_lat": lat,
            "gauge_easting": easting,
            "gauge_northing": northing,
            "gauge_elev_metadata": raw_meta["PNP"].values[0],
            "area_metadata": raw_meta["EZG_Gr"].values[0],
            "part_of_camelsp": False if meta_camelsp.empty else True
        }
    else:
        raise ValueError(f"Station {id} has no metadata.")
    
    # initialize the station
    station = Station1h(gauge_id)

    # save time series data
    station.save_data(data)

    # save metadata
    station.save_metadata(**station_meta)
    
    # save raw metadata
    station.save_raw_metadata(raw_meta)
    

100%|██████████| 47/47 [02:35<00:00,  3.30s/it]
